<a href="https://colab.research.google.com/github/MithunSrinivas28/EdgeVision/blob/main/final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Reset Keras/TensorFlow session


In [1]:
import tensorflow as tf
tf.keras.backend.clear_session()

Mount Google Drive to access project files

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Import required libraries

In [3]:
import tensorflow as tf
import numpy as np
import os
from sklearn.utils import class_weight
import shutil
import csv
from datetime import datetime
import pandas as pd

Define training and testing dataset locations

In [4]:
TRAIN_PATH = "/content/drive/MyDrive/Datasets/train"
TEST_PATH  = "/content/drive/MyDrive/Datasets/test"

Load training and testing datasets from Drive into TensorFlow format

In [5]:
train_data = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    image_size=(224,224),
    color_mode="grayscale",
    batch_size=32,
    shuffle=True
)

test_data = tf.keras.utils.image_dataset_from_directory(
    TEST_PATH,
    image_size=(224,224),
    color_mode="grayscale",
    batch_size=32,
    shuffle=True
)

Found 1032 files belonging to 8 classes.
Found 264 files belonging to 8 classes.


Retrieve and display class names from dataset

In [6]:
class_names = train_data.class_names
print(class_names)

['bridge', 'clean', 'cmp', 'crack', 'ler', 'open', 'others', 'vias']


Dataset preprocessing (normalize + convert grayscale to RGB)

In [7]:
def preprocess(image, label):
    image = image / 255.0                 # normalize
    image = tf.repeat(image, 3, axis=-1)  # gray → RGB
    return image, label

train_data = train_data.map(preprocess)
test_data  = test_data.map(preprocess)

Verify preprocessed image shape (sanity check)

In [8]:
for images, labels in train_data.take(1):
    print(images.shape)

(32, 224, 224, 3)


Loading a Pretrained Base Model (Feature Extractor)

In [9]:
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Freezing the pretrained model (Feature Extraction mode)

In [10]:
base_model.trainable = False

Building and Compiling the Final Classification Model

In [11]:
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 576)            │         2,304 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,016,312 (3.88 MB)

 Trainable params: 76,040 (297.03 KB)

 Non-trainable params: 940,272 (3.59 MB)

Computing class weights to handle imbalanced data

In [12]:
labels = np.concatenate([y for x,y in train_data], axis=0)

cw = class_weight.compute_class_weight(
    "balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(cw))
print("Class weights:", class_weights)

Class weights: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0), 5: np.float64(1.0), 6: np.float64(1.0), 7: np.float64(1.0)}


Training the model on the dataset

In [13]:
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=12
)

Epoch 1/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 106s 3s/step - accuracy: 0.2112 - loss: 1.9635 - val_accuracy: 0.1250 - val_loss: 2.1041
Epoch 2/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 26s 779ms/step - accuracy: 0.3547 - loss: 1.7386 - val_accuracy: 0.1553 - val_loss: 2.0751
Epoch 3/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 34s 562ms/step - accuracy: 0.4302 - loss: 1.6311 - val_accuracy: 0.1780 - val_loss: 2.0477
Epoch 4/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 26s 766ms/step - accuracy: 0.4651 - loss: 1.5322 - val_accuracy: 0.1780 - val_loss: 2.0162
Epoch 5/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 18s 556ms/step - accuracy: 0.4884 - loss: 1.4656 - val_accuracy: 0.2121 - val_loss: 1.9846
Epoch 6/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 19s 581ms/step - accuracy: 0.5455 - loss: 1.4048 - val_accuracy: 0.2348 - val_loss: 1.9442
Epoch 7/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 21s 584ms/step - accuracy: 0.5601 - loss: 1.3259 - val_accuracy: 0.3030 - val_loss: 1.8973
Epoch 8/12
33/33 ━━━━━━━━━━━━━━━━━━━━ 21s 624ms/step - accuracy: 0.5640 - loss: 1.3022 - val_accura

Fine-tuning / Refining the model with smaller learning rate + Early Stopping (Second Training Phase)

In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(2e-6),   # very gentle
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history_more = model.fit(
    train_data,
    validation_data=test_data,
    epochs=6,
    callbacks=[early]
)

Epoch 1/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 27s 614ms/step - accuracy: 0.6424 - loss: 1.1122 - val_accuracy: 0.6439 - val_loss: 1.5132
Epoch 2/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 21s 629ms/step - accuracy: 0.6318 - loss: 1.1293 - val_accuracy: 0.6932 - val_loss: 1.4546
Epoch 3/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 21s 620ms/step - accuracy: 0.6347 - loss: 1.1083 - val_accuracy: 0.7045 - val_loss: 1.3971
Epoch 4/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 20s 600ms/step - accuracy: 0.6308 - loss: 1.1018 - val_accuracy: 0.7273 - val_loss: 1.3411
Epoch 5/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 21s 623ms/step - accuracy: 0.6502 - loss: 1.0987 - val_accuracy: 0.7273 - val_loss: 1.2877
Epoch 6/6
33/33 ━━━━━━━━━━━━━━━━━━━━ 20s 597ms/step - accuracy: 0.6473 - loss: 1.0831 - val_accuracy: 0.7311 - val_loss: 1.2384


Final fine-tuning with ultra-small learning rate (Third training phase)

In [15]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-6),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_more = model.fit(
    train_data,
    validation_data=test_data,
    epochs=4,
    callbacks=[early]
)

Epoch 1/4
33/33 ━━━━━━━━━━━━━━━━━━━━ 28s 664ms/step - accuracy: 0.6153 - loss: 1.1210 - val_accuracy: 0.7386 - val_loss: 1.1938
Epoch 2/4
33/33 ━━━━━━━━━━━━━━━━━━━━ 20s 587ms/step - accuracy: 0.6453 - loss: 1.1039 - val_accuracy: 0.7348 - val_loss: 1.1551
Epoch 3/4
33/33 ━━━━━━━━━━━━━━━━━━━━ 19s 566ms/step - accuracy: 0.6047 - loss: 1.1317 - val_accuracy: 0.7311 - val_loss: 1.1222
Epoch 4/4
33/33 ━━━━━━━━━━━━━━━━━━━━ 20s 612ms/step - accuracy: 0.6405 - loss: 1.1070 - val_accuracy: 0.7311 - val_loss: 1.0954


Saving the trained model to Google Drive

In [16]:
model.save("/content/drive/MyDrive/wafer_xai3_model.keras")
print("Saved XAI model")

Saved XAI model


Listing convolutional layers inside the base model

In [17]:
for layer in model.layers[0].layers:
    if "conv" in layer.name.lower():
        print(layer.name)

conv
conv_bn
expanded_conv_depthwise_pad
expanded_conv_depthwise
expanded_conv_depthwise_bn
expanded_conv_squeeze_excite_avg_pool
expanded_conv_squeeze_excite_conv
expanded_conv_squeeze_excite_relu
expanded_conv_squeeze_excite_conv_1
expanded_conv_squeeze_excite_mul
expanded_conv_project
expanded_conv_project_bn
expanded_conv_1_expand
expanded_conv_1_expand_bn
expanded_conv_1_depthwise_pad
expanded_conv_1_depthwise
expanded_conv_1_depthwise_bn
expanded_conv_1_project
expanded_conv_1_project_bn
expanded_conv_2_expand
expanded_conv_2_expand_bn
expanded_conv_2_depthwise
expanded_conv_2_depthwise_bn
expanded_conv_2_project
expanded_conv_2_project_bn
expanded_conv_2_add
expanded_conv_3_expand
expanded_conv_3_expand_bn
expanded_conv_3_depthwise_pad
expanded_conv_3_depthwise
expanded_conv_3_depthwise_bn
expanded_conv_3_squeeze_excite_avg_pool
expanded_conv_3_squeeze_excite_conv
expanded_conv_3_squeeze_excite_relu
expanded_conv_3_squeeze_excite_conv_1
expanded_conv_3_squeeze_excite_mul
expande

Setting up Google Drive folder structure for the Wafer Pipeline project

In [18]:
BASE_DIR = "/content/drive/MyDrive/Wafer_Pipeline/Wafer_Pipeline"
MODEL_PATH = BASE_DIR + "/wafer_xai3_model.keras"
CSV_PATH = BASE_DIR + "/Results/predictions.csv"
SELF_LEARN_DIR = "/content/drive/MyDrive/Datasets/Self-learning"
os.makedirs(SELF_LEARN_DIR, exist_ok=True)
OFFLINE_SYNC_DIR = BASE_DIR + "/Offline_Storage/Pending_Sync"
os.makedirs(OFFLINE_SYNC_DIR, exist_ok=True)
SYNCED_DIR = BASE_DIR + "/Offline_Storage/Synced"
os.makedirs(SYNCED_DIR, exist_ok=True)
folders = [
    BASE_DIR,
    BASE_DIR + "/Input_Images",
    BASE_DIR + "/Results",
    BASE_DIR + "/Datasets/Self-learning",
    BASE_DIR + "/Offline_Storage/Pending_Sync"
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("Folder structure created")

Folder structure created


Preprocessing images for model prediction

In [19]:
# ===== IMAGE PREPROCESSING =====
IMG_SIZE = 224
def preprocess_image(img_path):
  img = tf.keras.utils.load_img(
        img_path,
        color_mode="grayscale",
        target_size=(IMG_SIZE, IMG_SIZE)
    )
  img = tf.keras.utils.img_to_array(img)
  img = img / 255.0 # Normalize
  img = tf.repeat(img, 3, axis=-1) # Gray → RGB
  img = tf.expand_dims(img, axis=0) # Add batch dimension
  return img

Function for single-image prediction

In [20]:
def predict_image(img_path):
    img = preprocess_image(img_path)
    preds = model.predict(img, verbose=0)[0]
    class_id = np.argmax(preds)
    confidence = float(np.max(preds))
    label = class_names[class_id]
    return label, confidence

Run prediction on a sample test image and display result

In [25]:
test_image = BASE_DIR + "/Input_Images/test1.png"
label, conf = predict_image(test_image)
print("Prediction :", label)
print("Confidence :", round(conf*100, 2), "%")

Prediction : open
Confidence : 35.1 %


Initialize results logging CSV file

In [26]:
def init_csv():
    if not os.path.exists(CSV_PATH):
        with open(CSV_PATH, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "Timestamp",
                "Image_Name",
                "Prediction",
                "Confidence",
                "Status"
            ])
        print("CSV file created")
init_csv()

CSV file created


User-facing wrapper to run inspection and display results

In [27]:
CONF_THRESHOLD=0.3
def inspect_image(img_path):
    img_name = os.path.basename(img_path)
    img = preprocess_image(img_path)
    preds = model.predict(img, verbose=0)[0]
    class_id = np.argmax(preds)
    confidence = float(np.max(preds))
    label = class_names[class_id]
    # ---- CHECK CONFIDENCE ----
    if confidence < CONF_THRESHOLD:
        status = "LOW_CONFIDENCE"
        shutil.copy(
            img_path,
            os.path.join(SELF_LEARN_DIR, img_name)
        )
        print("⚠ Sent to Self-learning folder")
    else:
        status = "OK"
    # ---- LOG TO CSV (OFFLINE HISTORY) ----
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(CSV_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            timestamp,
            img_name,
            label,
            round(confidence,4),
            status
        ])
    return label, confidence, status

Run full inspection on a sample image

In [29]:
test_image = BASE_DIR + "/Input_Images/test1.png"
inspect_image(test_image)

('open', 0.3509911000728607, 'OK')

View inspection log stored in CSV

In [30]:
pd.read_csv(CSV_PATH)

,Timestamp,Image_Name,Prediction,Confidence,Status
0,2026-11-01 19:31:40,test1.png,open,0.351,OK


Batch inspection of all images + automatic quality report

In [31]:
def run_batch_inspection():
    files = os.listdir(INPUT_DIR)
    if len(files) == 0:
        print("No images found in Input_Images folder")
        return
    print("Starting Batch Inspection...")
    print("Total Images:", len(files))
    print("-" * 40)
    defect_count = {}
    low_conf_count = 0
    for file in files:
        img_path = os.path.join(INPUT_DIR, file)
        # Skip non-image files
        if not file.lower().endswith((".png", ".jpg", ".jpeg")):
            continue
        label, conf, status = inspect_image(img_path)
        # Count defects
        defect_count[label] = defect_count.get(label, 0) + 1
        if status == "LOW_CONFIDENCE":
            low_conf_count += 1
        print(f"{file:20s} → {label:8s} ({conf*100:.1f}%) {status}")
    # ---- SUMMARY ----
    print("\nBatch Summary")
    print("-" * 40)
    total = sum(defect_count.values())
    for k, v in defect_count.items():
        print(f"{k:8s} : {v}")
    print("\nLow Confidence :", low_conf_count)
    yield_percent = (defect_count.get("clean", 0) / total) * 100
    print("Yield :", round(yield_percent, 2), "%")
    if yield_percent < 70:
        print("Batch Status : FAIL")
    else:
        print("Batch Status : PASS")

Sync offline images from Pending to Synced folder

In [32]:
def sync_data():
    pending_files = os.listdir(OFFLINE_SYNC_DIR)
    if len(pending_files) == 0:
        print("Nothing to sync")
        return
    print("Starting Sync...")
    print("-" * 30)
    synced_count = 0
    for file in pending_files:
        src = os.path.join(OFFLINE_SYNC_DIR, file)
        dst = os.path.join(SYNCED_DIR, file)
        # Avoid overwriting
        if os.path.exists(dst):
            continue
        shutil.move(src, dst)
        synced_count += 1
        print(f"Synced: {file}")
    print("-" * 30)
    print("Sync Complete")
    print("Files synced:", synced_count)

Run automated batch inspection on all input images

In [35]:
INPUT_DIR = BASE_DIR + "/Input_Images"
run_batch_inspection()
sync_data()

Starting Batch Inspection...
Total Images: 6
----------------------------------------
test1.png            → open     (35.1%) OK
⚠ Sent to Self-learning folder
test2.png            → vias     (20.7%) LOW_CONFIDENCE
cmp.png              → cmp      (51.4%) OK
test4.png            → crack    (37.5%) OK
test5.png            → open     (64.3%) OK
Screenshot 2026-01-30 165505.png → bridge   (53.1%) OK

Batch Summary
----------------------------------------
open     : 2
vias     : 1
cmp      : 1
crack    : 1
bridge   : 1

Low Confidence : 1
Yield : 0.0 %
Batch Status : FAIL
Nothing to sync


In [36]:
import os

MODEL_PATH = "/content/drive/MyDrive/wafer_xai3_model.keras"

size_mb = os.path.getsize(MODEL_PATH) / (1024*1024)
print(f"Model size: {round(size_mb, 2)} MB")


Model size: 5.01 MB
